In [1]:
import pandas as pd
from src.recovery_model import RecoveryModel

pd.set_option("multi_sparse", False)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
# Select a folder for the data to be used
folder = "test_1"  # choose between: test_1  / test_2  / test_fewer_layers / Toy_WEEE_v2
layer_0 = "flow"
layer_1 = "product"
layer_2 = "component"
layer_3 = "material"
layer_4 = "element"

In [3]:
# This section insures that the structure of the excel files is consistent

layer_names = (layer_0, layer_1, layer_2, layer_3, layer_4)

metadata = {
    "path": f"data/{folder}/",
    "filename": "metadata.csv",
}
composition = {
    "path": f"data/{folder}/",
    "filename": "composition.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Layer 1": layer_1,
        "Layer 2": layer_2,
        "Layer 3": layer_3,
        "Layer 4": layer_4,
        "Value": "data",
        "parameterCode": "parameterCode",
        "Year": "year",  # not considered at this stage
        "Scenario": "scenario",  # not considered at this stage
        "Location": "region",  # not considered at this stage
        "UoM": "unit",  # not considered at this stage
    },
    "parameterCode": {
        layer_2: "c-p",  # ! do not change value
        layer_3: "m-c",  # ! do not change value
        layer_4: "e-m",  # ! do not change value
    },
}

inputs = {
    "path": f"data/{folder}/",
    "filename": "inputs.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Substance_main_parent": layer_1,
        "Value": "data",
        "Unit": "unit",
        "Year": "year",  # not considered at this stage
    },
}

tcs = {
    "path": f"data/{folder}/",
    "filename": "TCs.csv",
    "mapper": {
        # inflows / outflows
        "input_flow": "inflows",
        "output_flow": "outflows",
        # levels
        "input_layer": "input_layer1_level",
        "input_sub_layer": "input_layer2_level",
        "output_layer": "output_layer1_level",
        # keys
        "input_layer_key": "input_layer1_key",
        "input_sub_layer_key": "input_layer2_key",
        "output_target_key": "output_layer1_key",
        # other columns
        "value": "data",
        "process": "process",
        "Year": "year",  # not considered at this stage
    },
    "all_symbols": {
        layer_1: "P*",
        layer_2: "C*",
        layer_3: "M*",
        layer_4: "E*",
    },
}

---


# Recovery model


In [4]:
model = RecoveryModel(
    name=folder,
    metadata=metadata,
    composition=composition,
    inputs=inputs,
    tcs=tcs,
    layer_names=layer_names,
    save_intermediary_steps=False,
)

---

# Metadata


In [5]:
model.dims

(8, 3, 3, 3, 3)

In [6]:
model.size

648

In [7]:
model.flows_eqs

,F1,F2,F3,F4,F5,F6,F7
process,,,,,,,
T1,1,-1,0,0,-1,0,0
T2,0,1,-1,-1,0,0,0
T3,0,0,1,0,0,-1,0
T4,0,0,0,1,1,0,-1


---

# Model equation

$$(I - A^T)x = y$$

<img src="./doc/img/model_equation.png" alt="model_equation" width="350" />

In [8]:
model.lneqs  # the A matrix (it will be properly renamed later)

<648x648 sparse matrix of type '<class 'numpy.float64'>'
	with 154 stored elements in Compressed Sparse Row format>

In [9]:
model.y

<648x1 sparse array of type '<class 'numpy.int64'>'
	with 2 stored elements in Compressed Sparse Column format>

---

# Solver


In [10]:
model.solve(aggregate=False, pivot=False).fillna("")

,flow,product,component,material,element,data
0,F1,P1,,,,1000.00
1,F1,P1,C1,,,800.00
2,F1,P1,C1,M1,,480.00
3,F1,P1,C1,M1,E1,240.00
4,F1,P1,C1,M1,E2,240.00
...,...,...,...,...,...,...
96,F6,P2,C1,M1,E1,0.97
97,F6,P2,C2,M2,E2,0.53
98,F7,P1,C1,M1,E1,39.19
99,F7,P1,C2,M1,E1,1.90


In [11]:
model.solve(aggregate=False, pivot=True).fillna("")

flow,product,component,material,element,F1,F2,F3,F4,F5,F6,F7
0,P1,,,,1000.00,,,,,,
1,P1,C1,,,800.00,328.00,,,,,
2,P1,C1,M1,,480.00,196.80,17.71,70.85,105.60,,
3,P1,C1,M1,E1,240.00,98.40,8.86,35.42,52.80,3.45,39.19
4,P1,C1,M1,E2,240.00,98.40,8.86,35.42,52.80,,
5,P1,C1,M2,,320.00,131.20,,,44.80,,
6,P1,C1,M2,E1,288.00,118.08,,,40.32,,
7,P1,C1,M2,E2,32.00,13.12,,,4.48,,
8,P1,C2,,,200.00,14.00,,,,,
9,P1,C2,M1,,40.00,2.80,,,8.80,,


---

# Mass balance

In [12]:
mass_balance = pd.read_csv(f"consolidation/{folder}_solution_mass_balance.csv", index_col=0)
mass_balance.fillna("")

,product,component,material,element,process,F1,F2,F3,F4,F5,F6,F7,mass_balance
0,P1,,,,T1,1000.00,,,,,,,1000.00
1,P1,C1,,,T1,800.00,-328.00,,,,,,472.00
2,P1,C1,M1,,T1,480.00,-196.80,0.00,0.00,-105.60,,,177.60
3,P1,C1,M1,E1,T1,240.00,-98.40,0.00,0.00,-52.80,0.00,0.00,88.80
4,P1,C1,M1,E2,T1,240.00,-98.40,0.00,0.00,-52.80,,,88.80
...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,P2,C1,M1,E1,T4,0.00,0.00,0.00,9.98,,0.00,-5.09,4.89
84,P2,C1,M1,E2,T4,0.00,0.00,0.00,14.97,,,,14.97
85,P2,C2,M2,,T4,0.00,0.00,0.00,9.41,,,,9.41
86,P2,C2,M2,E1,T4,0.00,0.00,0.00,7.53,,,,7.53


In [13]:
impossible_rows = mass_balance["mass_balance"] < 0
mass_balance[impossible_rows].fillna("")

,product,component,material,element,process,F1,F2,F3,F4,F5,F6,F7,mass_balance
